# Fast v4.2 → v3.1 antibody form converter (batch)
Streams input rows once to avoid the read-only Excel performance hang. Reuses `uploaded` in the current Colab runtime, or accepts multiple uploaded XLSX files. Saves each converted workbook immediately and creates a ZIP without triggering an automatic download.

Strict dropdown matching, row-2 headers, single-chain/two-chain isotype defaults, and plain-text Order Details are retained.


In [3]:
from google.colab import files
from openpyxl import Workbook, load_workbook
import io, zipfile, difflib, re, csv, os, time
OPTIONS={'Isotype': ['hIgG1', 'hIgG2', 'hIgG4', 'hIgG1 VHH Fc Fusion', 'hIgG2 VHH Fc Fusion', 'Other, please specify below'], 'Light Chain (λ or κ)': ['LAMBDA', 'KAPPA', 'N/A'], 'Leader Setting': ['PLEASE SELECT OPTION FROM DROPDOWN MENU', 'Twist Sequence', 'Contained In Insert Sequence', 'Fully Contained In Custom Vector', 'Partially Contained In Custom Vector'], 'Leader Sequence': ['PLEASE SELECT OPTION FROM DROPDOWN MENU', 'ATGAGAGCCTGGATCTTTTTCCTGCTGTGCCTGGCTGGGCGCGCCCTGGCC', 'Twist Vector: Leader fully contained in insert, ENTER SEQUENCE BELOW', 'Custom Vector: Leader fully contained in insert, ENTER SEQUENCE BELOW', 'Custom Vector: Vector fully contains Leader, ENTER SEQUENCE BELOW', 'Custom Vector: Leader partially contained in insert and vector, ENTER SEQUENCE BELOW'], 'Elution Buffer*': ['83 mM Glycine, 167 mM Tris-HCl pH 6 Elution Buffer', '43 mM Citrate, 147 mM HEPES pH 6 (Amine Free) Elution Buffer', '20 mM HEPES, 150 mM NaCl, pH 7.4 (his-tag default)'], 'Shipping Container*': ['2mL Tubes - tubes come with tube labels', '1.4 mL Micronic Tubes - tubes come with barcodes and plate map', '10 mL Tube (For 8mL SUP or 4 x 8mL Purified)', '96-well Deep-well Plate - comes with plate map', '4mL micronic (For 2-3 x 8mL Purified)'], 'Scale*': ['HT IgG 1 mL Supernatant', 'HT IgG 1 mL Purified', 'HT IgG 2 mL Purified', 'HT IgG 3 mL Purified', 'HT IgG 4 mL Purified', 'HT IgG 8 mL Supernatant ', 'HT IgG 8 mL Purified', 'HT IgG 16mL Purified', 'HT IgG 24mL Purified', 'HT IgG 32mL Purified'], 'Custom Plate Map': ['Customer provided custom plate map', 'No custom plate map'], 'Buffer Exchange': ['DPBS pH 7.4 Buffer Exchange', 'PBS pH 6.0 Buffer Exchange'], 'Normatlization': ['0.5 mg/mL Normalization', '1 mg/mL (For 8mL scale only) Normalization', 'No normalization'], 'Restriction Enzyme Sites to Avoid for DNA Optimization': ['Yes, please specify REs to avoid in Notes section to the right. -->', 'None'], 'Pooling': ['Pooled', 'No Pooling (Keep each expression in separate tubes)'], 'Kozak Setting': ['PLEASE SELECT OPTION FROM DROPDOWN MENU', 'Twist Sequence', 'Contained In Insert Sequence', 'Fully Contained In Custom Vector'], 'Kozak Sequence': ['GCCGCCACC', 'GCCACC', 'Custom: Kozak fully contained in insert, ENTER SEQUENCE', 'Custom: Vector fully contained Kozak, ENTER SEQUENCE'], 'VH Vector Name - Insertion Site Name': ['PLEASE SELECT OPTION FROM DROPDOWN MENU', 'pTwist CMV hIgG2-HuIgG2Fc_TAG', 'pTwist CMV hIgG1-Fc-VHHFc_TAG', 'pTwist CMV hIgG1-HuIgG1Fc_TAG', 'pTwist CMV BG WPRE Neo-NotI_XbaI', 'pTwist CMV BG WPRE Neo-XhoI_ClaI', 'pTwist CMV BG WPRE Neo-EcoRI_XbaI', 'pTwist CMV BG WPRE Neo-XhoI_XbaI', 'pTwist CMV BG WPRE Neo-NotI_ClaI', 'pTwist CMV BG WPRE Neo-EcoRI_ClaI', 'pTwist CMV hIgG4 S228P-HuIgG4_TAG', 'pTwist CMV hIgG2-Fc-NotI_GS-hIgG2-FcFusionTA', 'pTwist CMV mIgG2a (truncated) - VHH mIgG2a', 'pTwist CMV rbIgG1-rbIgG1', 'pTwist CMV mIgG2a-mIgG2a', 'pTwist CMV hIgG1 - M428L/N434S-hIgG1 M428L/N434S', 'pTwist CMV hIgG1 - LALAPG + YTE-hIgG1L234A_L235A+P329G+M252Y_S254T_T256E', 'Custom - ENTER VECTOR NAME - INSERTION SITE'], 'VL Vector Name - Insertion Site Name': ['PLEASE SELECT OPTION FROM DROPDOWN MENU', 'pTwist CMV hIgL2-Lambda_TAG', 'pTwist CMV hIgK-Kappa_TAG', 'pTwist CMV BG WPRE Neo-NotI_XbaI', 'pTwist CMV BG WPRE Neo-XhoI_ClaI', 'pTwist CMV BG WPRE Neo-EcoRI_XbaI', 'pTwist CMV BG WPRE Neo-XhoI_XbaI', 'pTwist CMV BG WPRE Neo-NotI_ClaI', 'pTwist CMV BG WPRE Neo-EcoRI_ClaI', 'pTwist CMV rbIgk2-rbIgk2', 'pTwist CMV rbIgk1-rbIgk1', 'pTwist CMV mIgL2-mIgL2', 'pTwist CMV mIgL1-mIgL1', 'pTwist CMV mIgk-mIgk', 'Custom - ENTER VECTOR NAME - INSERTION SITE'], 'Purification': ['Protein A purification', 'Ni IMAC (His-tag) '], 'Please indicate if you prefer not to receive physical samples and only data': ['No physical shipment'], 'Post Purification Sterile Filtration': ['Post Purification Sterile Filtration'], 'Aliquots': ['2 aliquots (1mL Pur)', '2 aliquots (8mL Pur)', '3 aliquots (8mL Pur)', '4 aliquots (8mL Pur)', '2 aliquots (1mL Sup)', '4 aliquots (1mL Sup)', '2 aliquots (8mL Sup)', '4 aliquots (8mL Sup)', 'Custom (specify to the right)  '], 'Ship with testing pending': ['Yes', 'No']}
MAP={"C":"C","G":"D","H":"E","I":"F","J":"G","K":"H","L":"I","M":"J","N":"K","O":"L","P":"M","Q":"N","R":"O","S":"P","T":"Q","U":"R","V":"S"}
HEADERS=['Plate #', 'Well', 'IgG Name', 'Isotype', 'Light Chain (LAMBDA or KAPPA)', 'VH Name', 'VH Vector Name - Insertion Site Name', 'VH Kozak Settings', 'VH Kozak Sequence (DNA)', 'VH Leader Setting', 'VH Leader Sequence (DNA)', 'VH (VDJ) Sequence (DNA or AA)', 'VL Name', 'VL Vector Name - Insertion Site Name', 'VL Kozak Settings', 'VL Kozak Sequence (DNA)', 'VL Leader Setting', 'VL Leader Sequence (DNA)', 'VL (VJ) Sequence (DNA or AA)']
ORDER=[['Sequence Intake Form', 'HT IgG Production', 'Contact: igg@twistbioscience.com'], ['Order Details (* = Required)', 'Please Select an Option', 'Notes'], ['Scale*', 'HT IgG 4 mL Purified', ''], ['Pooling (Only applies to multiple 1mL or 8mL scales)', 'Pooled', ''], ['Cell Line*', 'HEK293', ''], ['Purification Method*', 'Protein A purification', ''], ['Elution Buffer*', '43 mM Citrate, 147 mM HEPES pH 6 (Amine Free) Elution Buffer', ''], ['Buffer Exchange', 'DPBS pH 7.4 Buffer Exchange', ''], ['Normalization', 'No normalization', ''], ['Post Purification Sterile Filtration', '', ''], ['Aliquoting (Not applicable when characterization services and testing are added)', '', ''], ['Shipping Container*', '96-well Deep-well Plate - comes with plate map', ''], ['Custom Plate Map', '', ''], ['Restriction Enzyme Sites to Avoid for Codon Optimization', '', ''], ['Please indicate if you prefer not to receive any physical samples and only data', '', ''], ['Ship physical samples with additional service requests pending (Additional material will need to be pulled and held on reserve if selected)', '', ''], ['Additional Requests (Subject to feasibility review)', '', '']]
GROUPS={"D":"Isotype","E":"Light Chain (λ or κ)","G":"VH Vector Name - Insertion Site Name","H":"Kozak Setting","I":"Kozak Sequence","J":"Leader Setting","K":"Leader Sequence","N":"VL Vector Name - Insertion Site Name","O":"Kozak Setting","P":"Kozak Sequence","Q":"Leader Setting","R":"Leader Sequence"}

# Exact choices from the supplied v3.1 template.
GROUPS={"E":"Light Chain (λ or κ)","G":"VH Vector Name - Insertion Site Name","H":"Kozak Setting","I":"Kozak Sequence","J":"Leader Setting","K":"Leader Sequence","N":"VL Vector Name - Insertion Site Name","O":"Kozak Setting","P":"Kozak Sequence","Q":"Leader Setting","R":"Leader Sequence"}

def match(value, category, custom_vector=False):
    if value is None or str(value).strip()=="": return "", "blank"
    value=str(value).strip()
    choices=[x for x in OPTIONS[category] if not x.startswith("PLEASE SELECT")]
    if value in choices: return value,"exact"
    stripped=re.sub(r"^Empty Vector\s*-\s*","",value,flags=re.I)
    if stripped in choices: return stripped,"normalized"
    scores=sorted(((difflib.SequenceMatcher(None,stripped.lower(),x.lower()).ratio(),x)
                   for x in choices if not x.startswith("Custom -")),reverse=True)
    if scores and scores[0][0]>=.90 and (len(scores)==1 or scores[0][0]-scores[1][0]>=.04):
        return scores[0][1],"fuzzy"
    return (value,"custom vector") if custom_vector else ("","INVALID DROPDOWN")

def convert(blob):
    src=load_workbook(io.BytesIO(blob),read_only=True,data_only=True)
    try:
        if "Antibody Designs" not in src.sheetnames: raise ValueError("Missing Antibody Designs tab")
        sheet=src["Antibody Designs"]
        wb=Workbook(); dest=wb.active; dest.title="Sequence"
        dest.append([""]*19)  # row 1 intentionally blank
        dest.append(HEADERS)  # headers in row 2
        issues=[]; total=0
        # Critical speed fix: stream rows ONCE, never repeated random read-only cell access.
        for i,row in enumerate(sheet.iter_rows(min_row=4,max_col=22,values_only=True),start=4):
            if not row[2]: continue
            values={dst:(row[ord(col)-65] or "") for col,dst in MAP.items()}
            two_chain=bool(str(values.get("S","")).strip() or str(values.get("M","")).strip())
            values["D"]="hIgG1" if two_chain else "hIgG1 VHH Fc Fusion"
            if not two_chain: values["E"]="N/A"
            for col,category in GROUPS.items():
                original=values.get(col,"")
                value,status=match(original,category,col in ("G","N"))
                if status=="INVALID DROPDOWN": issues.append((i,col,original,"cleared: not in dropdown"))
                values[col]=value
            dest.append([values.get(c,"") for c in "ABCDEFGHIJKLMNOPQRS"])
            total+=1
        details=wb.create_sheet("Order Details")
        for row in ORDER: details.append(row)
        out=io.BytesIO(); wb.save(out)
        return out.getvalue(),total,issues
    finally:
        src.close()

def run_batch(uploaded):
    os.makedirs("/content/converted",exist_ok=True)
    report=[]; output_paths=[]
    for index,(name,blob) in enumerate(uploaded.items(),1):
        if not name.lower().endswith(".xlsx"): continue
        print(f"[{index}/{len(uploaded)}] Starting {name}",flush=True)
        start=time.monotonic()
        try:
            content,count,issues=convert(blob)
            filename=os.path.splitext(os.path.basename(name))[0]+"_v3.1.xlsx"
            path=os.path.join("/content/converted",filename)
            with open(path,"wb") as f:f.write(content)
            output_paths.append(path)
            report.append([name,count,len(issues),"OK",""])
            print(f"  DONE: {count} entries, {len(issues)} warnings, {time.monotonic()-start:.1f}s",flush=True)
            for row,col,value,reason in issues:
                warning_rows.append([name,row,col,value,reason])
        except Exception as exc:
            report.append([name,0,0,"ERROR",str(exc)])
            print(f"  ERROR: {exc}",flush=True)
    archive="/content/v4.2_to_v3.1_FAST.zip"
    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
        for path in output_paths:z.write(path,arcname=os.path.basename(path))
        for filename,header,entries in [("conversion_report.csv",["Input","Antibodies","Warnings","Status","Error"],report),("dropdown_warnings.csv",["Input","Row","Column","Original","Action"],warning_rows)]:
            f=io.StringIO(); w=csv.writer(f);w.writerow(header);w.writerows(entries);z.writestr(filename,f.getvalue())
    with zipfile.ZipFile(archive) as z: assert z.testzip() is None
    print(f"FINISHED: {len(output_paths)} files. ZIP: {archive} ({os.path.getsize(archive)/1e6:.2f} MB)")
    print("Download from Colab Files sidebar; no automatic browser download.")
    return archive

# Reuse files already uploaded in this Colab session if available.
if "uploaded" not in globals() or not uploaded:
    print("Select or drag multiple XLSX files into the upload picker.")
    uploaded=files.upload()
warning_rows=[]
if uploaded: output_zip=run_batch(uploaded)
else: print("No files supplied.")


[1/1] Starting Antibodies custom form v4.2_Amazon_lGDF8_2.xlsx
  DONE: 257 entries, 0 warnings, 0.5s
FINISHED: 1 files. ZIP: /content/v4.2_to_v3.1_FAST.zip (0.04 MB)
Download from Colab Files sidebar; no automatic browser download.
